# Apex3D · 01b · ONNX export and INT8 quantization

**Owner:** Person A · P5 §4.6 requires at least one optimisation technique.

We export the notebook-01 checkpoint to ONNX, then compare four variants **on the full test set**, using the same box crops and preprocessing:
PyTorch FP32 (reference), ONNX FP32, ONNX FP16 (reduced precision), and ONNX INT8 with dynamic and with static (calibrated) quantization.
Latency is batch 1 on the CPU with ONNX Runtime, so recognition can run while the GPU generates 3D meshes.

In [1]:
import os, sys, json, time
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "cls_lib.py").exists())
sys.path.insert(0, str(REPO / "notebooks"))
import numpy as np, torch, onnx, onnxruntime as ort
from PIL import Image
import cls_lib as L

BUNDLE = REPO / "checkpoints" / "apex3d_convnext_tiny.pt"
OUT = REPO / "checkpoints"; RESULTS = REPO / "results"
split_test = L.load_split(L.find_stanford_root(Path(os.environ.get("APEX3D_DATA", REPO / "data" / "raw"))), "test")
CACHE = REPO / "data" / "crops"
te_paths = [str(CACHE / (Path(r.path).parent.name + "_" + Path(r.path).stem + ".jpg")) for r in split_test]
te_y = np.array([r.label for r in split_test])
assert all(Path(p).exists() for p in te_paths), "run notebook 01 first (it caches the crops)"
clf = L.load_classifier(BUNDLE, "cpu")
print("onnx", onnx.__version__, "| onnxruntime", ort.__version__, ort.get_available_providers())

onnx 1.23.2 | onnxruntime 1.31.1 ['AzureExecutionProvider', 'CPUExecutionProvider']


## Export to ONNX (FP32) and verify against PyTorch

In [2]:
S = clf.img_size
fp32_path = OUT / "apex3d_convnext_tiny_fp32.onnx"
dummy = torch.randn(1, 3, S, S)
torch.onnx.export(clf.model, dummy, fp32_path, input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
onnx.checker.check_model(onnx.load(fp32_path))
sess = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
x = torch.randn(4, 3, S, S)
diff = np.abs(sess.run(None, {"input": x.numpy()})[0] - clf.model(x).detach().numpy()).max()
print(f"max |ONNX - PyTorch| logit difference: {diff:.2e}")
assert diff < 1e-3

C:\Users\Shreyaskm\AppData\Local\Temp\ipykernel_41436\2561547473.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(clf.model, dummy, fp32_path, input_names=["input"], output_names=["logits"],


max |ONNX - PyTorch| logit difference: 1.26e-05


## Reduced-precision and INT8 variants

- **FP16:** weights and activations in half precision (`onnxconverter-common`), the proposal's fallback.
- **INT8 dynamic:** MatMul/Gemm weights stored as 8-bit integers, activations quantized on the fly at run time. No calibration data needed.
- **INT8 static (QDQ), all layers:** weights *and* activations in INT8, per-channel weight scales, with activation ranges calibrated (min-max) on 256 crops from the **training split**.
- **INT8 static (QDQ), MatMul only:** the same, but only the pointwise MLP MatMuls (most of the compute and parameters) are quantized. The depthwise 7×7 convolutions, downsampling layers and classifier stay FP32. ConvNeXt has large activation outliers that min-max calibration handles badly in the other layers.

LayerNorm and GELU stay in floating point in every variant. The variant the app should deploy is chosen on **validation** (next sections), never on test.

In [3]:
from onnxruntime.quantization import quantize_dynamic, quantize_static, QuantType, QuantFormat, CalibrationDataReader
from onnxruntime.quantization.shape_inference import quant_pre_process
pre_path = OUT / "apex3d_convnext_tiny_pre.onnx"
quant_pre_process(str(fp32_path), str(pre_path))

int8_dyn_path = OUT / "apex3d_convnext_tiny_int8_dynamic.onnx"
quantize_dynamic(str(pre_path), str(int8_dyn_path), weight_type=QuantType.QInt8)

split = json.loads((RESULTS / "split.json").read_text()) if (RESULTS / "split.json").exists() else json.loads((REPO / "results" / "split.json").read_text())
calib_files = [CACHE / ("cars_train_" + Path(f).stem + ".jpg") for f in np.random.default_rng(0).permutation(split["train"])]
calib_files = [p for p in calib_files if p.exists()][:256]
print("calibration crops (train split):", len(calib_files))
class TrainCalib(CalibrationDataReader):
    def __init__(self): self.it = iter(calib_files)
    def get_next(self):
        p = next(self.it, None)
        return None if p is None else {"input": clf.transform(Image.open(p).convert("RGB")).unsqueeze(0).numpy()}
int8_static_path = OUT / "apex3d_convnext_tiny_int8_static_all.onnx"
quantize_static(str(pre_path), str(int8_static_path), TrainCalib(), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
int8_mm_path = OUT / "apex3d_convnext_tiny_int8_static_matmul.onnx"
quantize_static(str(pre_path), str(int8_mm_path), TrainCalib(), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, op_types_to_quantize=["MatMul"])
pre_path.unlink()

import warnings
from onnxconverter_common import float16
fp16_path = OUT / "apex3d_convnext_tiny_fp16.onnx"
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # tiny weights truncated to FP16's range; expected
        onnx.save(float16.convert_float_to_float16(onnx.load(fp32_path), keep_io_types=True), fp16_path)
    ort.InferenceSession(str(fp16_path), providers=["CPUExecutionProvider"])   # loads on this CPU?
except Exception as e:                       # reported, not hidden: the table will lack FP16
    print("FP16 variant unavailable on this runtime:", e); fp16_path = None
for p in (fp32_path, fp16_path, int8_dyn_path, int8_static_path, int8_mm_path):
    if p: print(f"{p.name}: {p.stat().st_size/1e6:.1f} MB")

calibration crops (train split): 256


apex3d_convnext_tiny_fp32.onnx: 112.0 MB
apex3d_convnext_tiny_fp16.onnx: 56.0 MB
apex3d_convnext_tiny_int8_dynamic.onnx: 28.4 MB
apex3d_convnext_tiny_int8_static_all.onnx: 28.7 MB
apex3d_convnext_tiny_int8_static_matmul.onnx: 34.6 MB


## CPU thread count

This laptop's CPU mixes fast performance cores with slower efficiency cores. ONNX Runtime's default uses all physical cores, and the slow cores then hold up every layer. So we measure FP32 latency at several thread counts, and use the fastest for every variant (PyTorch included).

In [4]:
x1 = np.random.randn(1, 3, S, S).astype(np.float32)
def session(path, threads):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
def latency(run, n=50):
    for _ in range(10): run(x1)
    t0 = time.perf_counter()
    for _ in range(n): run(x1)
    return (time.perf_counter() - t0) / n * 1000
sweep = {}
for th in (0, 2, 4, 6, 8):
    s_ = session(fp32_path, th); sweep[th] = round(latency(lambda x: s_.run(None, {"input": x})[0], n=30), 2)
    print(f"threads={th or 'default'}: {sweep[th]} ms", flush=True)
THREADS = min((k for k in sweep if k), key=sweep.get)
torch.set_num_threads(THREADS)
print("using", THREADS, "threads")

threads=default: 152.36 ms


threads=2: 55.91 ms


threads=4: 39.42 ms


threads=6: 37.3 ms


threads=8: 40.0 ms


using 6 threads


## Accuracy (validation and full test set) and CPU latency

Validation (1,222 images) is used to choose the variant for the app. Test (8,041 images) is reported for every variant, once.

In [5]:
train_lab = {Path(r.path).name: r.label for r in L.load_split(L.find_stanford_root(Path(os.environ.get("APEX3D_DATA", REPO / "data" / "raw"))), "train")}
va_files = [f for f in split["val"] if (CACHE / ("cars_train_" + Path(f).stem + ".jpg")).exists()]
va_paths = [str(CACHE / ("cars_train_" + Path(f).stem + ".jpg")) for f in va_files]
va_y = np.array([train_lab[f] for f in va_files])

def batches(paths, bs=64):
    for i in range(0, len(paths), bs):
        yield np.stack([clf.transform(Image.open(p).convert("RGB")).numpy() for p in paths[i:i + bs]])

def evaluate(run, paths, ys):
    logits = np.concatenate([run(xb) for xb in batches(paths)])
    top5 = np.argsort(-logits, 1)[:, :5]
    return logits, float((top5[:, 0] == ys).mean()), float((top5 == ys[:, None]).any(1).mean())

variants = {"pytorch_fp32": (lambda xb: clf.model(torch.from_numpy(xb)).detach().numpy(), BUNDLE)}
for name, path in (("onnx_fp32", fp32_path), ("onnx_fp16", fp16_path), ("onnx_int8_dynamic", int8_dyn_path),
                   ("onnx_int8_static_all", int8_static_path), ("onnx_int8_static_matmul", int8_mm_path)):
    if path is None: continue
    variants[name] = ((lambda s: lambda xb: s.run(None, {"input": xb})[0])(session(path, THREADS)), path)

rows, ref_pred = {}, None
for name, (run, path) in variants.items():
    with torch.no_grad():
        _, v1, v5 = evaluate(run, va_paths, va_y)
        t0 = time.time(); logits, t1, t5 = evaluate(run, te_paths, te_y); ev_s = time.time() - t0
        lat = latency(run)
    pred = logits.argmax(1)
    if ref_pred is None: ref_pred = pred
    rows[name] = {"size_mb": round(path.stat().st_size / 1e6, 2), "val_top1": v1, "val_top5": v5, "test_top1": t1, "test_top5": t5,
                  "agreement_with_pytorch_fp32": float((pred == ref_pred).mean()), "cpu_latency_ms_b1": round(lat, 2), "eval_seconds": round(ev_s, 1)}
    print(name, rows[name], flush=True)

pytorch_fp32 {'size_mb': 111.95, 'val_top1': 0.9214402618657938, 'val_top5': 0.9885433715220949, 'test_top1': 0.9220246238030095, 'test_top5': 0.9842059445342619, 'agreement_with_pytorch_fp32': 1.0, 'cpu_latency_ms_b1': 49.2, 'eval_seconds': 286.4}


onnx_fp32 {'size_mb': 111.96, 'val_top1': 0.9214402618657938, 'val_top5': 0.9885433715220949, 'test_top1': 0.9220246238030095, 'test_top5': 0.9842059445342619, 'agreement_with_pytorch_fp32': 1.0, 'cpu_latency_ms_b1': 34.33, 'eval_seconds': 323.7}


onnx_fp16 {'size_mb': 56.05, 'val_top1': 0.9214402618657938, 'val_top5': 0.9885433715220949, 'test_top1': 0.9219002611615471, 'test_top5': 0.9842059445342619, 'agreement_with_pytorch_fp32': 0.9998756373585375, 'cpu_latency_ms_b1': 38.61, 'eval_seconds': 395.8}


onnx_int8_dynamic {'size_mb': 28.36, 'val_top1': 0.9222585924713584, 'val_top5': 0.986088379705401, 'test_top1': 0.9197860962566845, 'test_top5': 0.9844546698171869, 'agreement_with_pytorch_fp32': 0.9896779007586121, 'cpu_latency_ms_b1': 151.34, 'eval_seconds': 4043.4}


onnx_int8_static_all {'size_mb': 28.69, 'val_top1': 0.839607201309329, 'val_top5': 0.9664484451718495, 'test_top1': 0.8610869294863823, 'test_top5': 0.9665464494465863, 'agreement_with_pytorch_fp32': 0.8821042158935456, 'cpu_latency_ms_b1': 21.59, 'eval_seconds': 216.4}


onnx_int8_static_matmul {'size_mb': 34.59, 'val_top1': 0.9198036006546645, 'val_top5': 0.9885433715220949, 'test_top1': 0.9217758985200846, 'test_top5': 0.9850764830244995, 'agreement_with_pytorch_fp32': 0.9950254943414998, 'cpu_latency_ms_b1': 19.4, 'eval_seconds': 250.6}


In [6]:
import pandas as pd
df = pd.DataFrame(rows).T; display(df)
base = rows["onnx_fp32"]
summary = {}
for q in [v for v in ("onnx_int8_dynamic", "onnx_int8_static_all", "onnx_int8_static_matmul", "onnx_fp16") if v in rows]:
    summary[q] = {"size_reduction_vs_onnx_fp32": 1 - rows[q]["size_mb"] / base["size_mb"],
                  "speedup_vs_onnx_fp32": base["cpu_latency_ms_b1"] / rows[q]["cpu_latency_ms_b1"],
                  "top1_change_vs_onnx_fp32": rows[q]["test_top1"] - base["test_top1"]}
    print(q, {k: round(v, 4) for k, v in summary[q].items()})

# Deployment choice, on VALIDATION only: the fastest ONNX variant within 0.5 points of ONNX FP32 val Top-1.
eligible = [k for k in rows if k.startswith("onnx") and rows[k]["val_top1"] >= base["val_top1"] - 0.005]
DEPLOY = min(eligible, key=lambda k: rows[k]["cpu_latency_ms_b1"])
print("deploy for CPU recognition:", DEPLOY, rows[DEPLOY])
out = {"generated_by": "notebooks/01b_export_quantize.ipynb", "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
       "cpu": os.environ.get("PROCESSOR_IDENTIFIER", ""), "threads": THREADS, "thread_sweep_fp32_ms": sweep, "onnxruntime": ort.__version__,
       "calibration": {"images": len(calib_files), "source": "training split crops"},
       "val_images": int(len(va_y)), "test_images": int(len(te_y)), "variants": rows, "summary": summary,
       "deploy_variant": DEPLOY, "deploy_rule": "fastest ONNX variant with val Top-1 within 0.5 points of ONNX FP32 (chosen on validation)"}
(RESULTS / "optimization_metrics.json").write_text(json.dumps(out, indent=2))
print("saved results/optimization_metrics.json")

,size_mb,val_top1,val_top5,test_top1,test_top5,agreement_with_pytorch_fp32,cpu_latency_ms_b1,eval_seconds
pytorch_fp32,111.95,0.921440,0.988543,0.922025,0.984206,1.000000,49.20,286.4
onnx_fp32,111.96,0.921440,0.988543,0.922025,0.984206,1.000000,34.33,323.7
onnx_fp16,56.05,0.921440,0.988543,0.921900,0.984206,0.999876,38.61,395.8
onnx_int8_dynamic,28.36,0.922259,0.986088,0.919786,0.984455,0.989678,151.34,4043.4
onnx_int8_static_all,28.69,0.839607,0.966448,0.861087,0.966546,0.882104,21.59,216.4
onnx_int8_static_matmul,34.59,0.919804,0.988543,0.921776,0.985076,0.995025,19.40,250.6


onnx_int8_dynamic {'size_reduction_vs_onnx_fp32': 0.7467, 'speedup_vs_onnx_fp32': 0.2268, 'top1_change_vs_onnx_fp32': -0.0022}
onnx_int8_static_all {'size_reduction_vs_onnx_fp32': 0.7437, 'speedup_vs_onnx_fp32': 1.5901, 'top1_change_vs_onnx_fp32': -0.0609}
onnx_int8_static_matmul {'size_reduction_vs_onnx_fp32': 0.6911, 'speedup_vs_onnx_fp32': 1.7696, 'top1_change_vs_onnx_fp32': -0.0002}
onnx_fp16 {'size_reduction_vs_onnx_fp32': 0.4994, 'speedup_vs_onnx_fp32': 0.8891, 'top1_change_vs_onnx_fp32': -0.0001}
deploy for CPU recognition: onnx_int8_static_matmul {'size_mb': 34.59, 'val_top1': 0.9198036006546645, 'val_top5': 0.9885433715220949, 'test_top1': 0.9217758985200846, 'test_top5': 0.9850764830244995, 'agreement_with_pytorch_fp32': 0.9950254943414998, 'cpu_latency_ms_b1': 19.4, 'eval_seconds': 250.6}
saved results/optimization_metrics.json
